# Partial derivatives and the gradient

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Functions of several variables
The loss of a model depends on **all of its weights**: $L(w_1,w_2,\dots,w_n)$. A large network has millions of them. We want to know how changing **each one** affects the loss.

## Partial derivatives
The partial derivative with respect to $x$ means: **hold the other variables fixed** and differentiate with respect to $x$ only. Notation: $\dfrac{\partial f}{\partial x}$.

Example: $f(x,y)=x^2y+3y$

| Derivative | Hold fixed | Result | At the point $(2,1)$ |
|---|---|---|---|
| $\partial f/\partial x$ | y is a constant | $2xy$ | $2\times2\times1=4$ |
| $\partial f/\partial y$ | x is a constant | $x^2+3$ | $4+3=7$ |

## The gradient
The gradient is the **vector of all partial derivatives**:

$$\nabla f=\left(\frac{\partial f}{\partial x_1},\ \frac{\partial f}{\partial x_2},\ \dots,\ \frac{\partial f}{\partial x_n}\right)$$

In the example above $\nabla f(2,1)=(4,7)$.

> [The orange arrows are gradients: always perpendicular to the contour lines and pointing uphill.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_gradient.png)

*The orange arrows are gradients: always perpendicular to the contour lines and pointing uphill.*

Three key facts about the gradient:

| Property | Meaning |
|---|---|
| Direction | The fastest **increase** of the function |
| Size $\lVert\nabla f\rVert$ | How steep it is in that direction |
| $\nabla f=0$ | A critical point: minimum, maximum or saddle |

> Because we want the loss to go **down**, we move along the **negative gradient**. That is the whole idea of gradient descent.

## Worked example: $f(x,y)=x^2+2y^2$ at $(1,1)$
| Quantity | Calculation | Value |
|---|---|---|
| $\partial f/\partial x$ | $2x$ | 2 |
| $\partial f/\partial y$ | $4y$ | 4 |
| $\nabla f$ | | $(2,4)$ |
| $\lVert\nabla f\rVert$ | $\sqrt{4+16}$ | 4.47 |
| One step with $\eta=0.1$ | $(1,1)-0.1(2,4)$ | $(0.8,\ 0.6)$ |
| Function value before and after | $3\to0.64+0.72$ | $3\to1.36$ |

## The gradient of the linear-regression loss
Model $\hat{y}=wx+b$ and loss $L=\dfrac1n\sum(\hat{y}_i-y_i)^2$:

$$\frac{\partial L}{\partial w}=\frac2n\sum(\hat{y}_i-y_i)\,x_i\qquad\frac{\partial L}{\partial b}=\frac2n\sum(\hat{y}_i-y_i)$$

With the data $(1,2),(2,4),(3,5)$ and starting from $w=1,\ b=0$:

| $x_i$ | $y_i$ | $\hat{y}_i$ | $\hat{y}_i-y_i$ | $(\hat{y}_i-y_i)x_i$ |
|---|---|---|---|---|
| 1 | 2 | 1 | −1 | −1 |
| 2 | 4 | 2 | −2 | −4 |
| 3 | 5 | 3 | −2 | −6 |
| Sum | | | −5 | −11 |

So $L=(1+4+4)/3=3$, $\partial L/\partial w=\frac23(-11)=-7.33$ and $\partial L/\partial b=\frac23(-5)=-3.33$. Both are negative: we must **increase** w and b.

## Jacobian and Hessian
| Name | What it is | Shape | Used for |
|---|---|---|---|
| Gradient $\nabla f$ | First derivatives of a scalar function | Vector of n | Gradient descent |
| Jacobian $J$ | First derivatives of a vector function | $m\times n$ matrix | Backpropagation through layers |
| Hessian $H$ | Second derivatives | $n\times n$ matrix | Curvature; Newton's method; testing for a minimum |

For $f=x^2+2y^2$ the Hessian is $\begin{bmatrix}2&0\\0&4\end{bmatrix}$. Positive eigenvalues (2 and 4) mean that the critical point $(0,0)$ is a **minimum**. One positive and one negative would give a **saddle point**.



In [ ]:
import numpy as np

def numeric_gradient(f, p, h=1e-5):
    """Central differences, one coordinate at a time."""
    p = np.array(p, dtype=float); g = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p); e[i] = h
        g[i] = (f(p + e) - f(p - e)) / (2 * h)
    return g

f1 = lambda p: p[0] ** 2 * p[1] + 3 * p[1]
print("grad of x^2 y + 3y at (2, 1):", numeric_gradient(f1, [2, 1]).round(4), " exact (4, 7)")

f2 = lambda p: p[0] ** 2 + 2 * p[1] ** 2
g = numeric_gradient(f2, [1, 1])
print("grad of x^2 + 2y^2 at (1, 1):", g.round(4), "  size:", round(np.linalg.norm(g), 3))
new = np.array([1, 1]) - 0.1 * g
print("after one step:", new.round(3), "  f:", f2([1, 1]), "->", round(f2(new), 3))

# Gradient of the MSE loss for y_hat = w x + b
x = np.array([1.0, 2.0, 3.0]); y = np.array([2.0, 4.0, 5.0])
def loss(p):
    return np.mean((p[0] * x + p[1] - y) ** 2)
w, b = 1.0, 0.0
err = w * x + b - y
print("loss:", loss([w, b]))
print("formula  dL/dw, dL/db:", round(2 * np.mean(err * x), 3), round(2 * np.mean(err), 3))
print("numeric  dL/dw, dL/db:", numeric_gradient(loss, [w, b]).round(3))

# Hessian eigenvalues tell the kind of critical point
for name, H in [("x^2 + 2y^2", [[2, 0], [0, 4]]), ("x^2 - y^2 ", [[2, 0], [0, -2]])]:
    ev = np.linalg.eigvalsh(np.array(H, dtype=float))
    print(name, "eigenvalues", ev, "->", "minimum" if ev.min() > 0 else "saddle point")

### Exercise

Compute the gradient of $f(x,y)=3x^2-2xy+y^2$ at $(1,2)$ by hand and check with code.

In [ ]:
# Your answer here